In [ ]:
pip install transformers scikit-learn torch numpy pandas anthropic

In [ ]:
#Load packages, data and model

import os
from anthropic import Anthropic
import pandas as pd
import numpy as np
import sklearn as sk
import json
import re
from typing import List, Dict, Any
import time

#Initialize Claude client
client = Anthropic(
    api_key= "INSERT_KEY"  
)

#Latest model name for Claude Sonnet 4 on
model_name = "claude-sonnet-4-20250514"  #Latest Sonnet model on August 13th 2025

#unlabelled dataset
df = pd.read_csv("../unlabelled_relevance.csv")

#labelled_dataset from Annotator 1 in round 2
df_gold = pd.read_csv('../Human/A1_r2.csv')

with open('../relevance_codebook.json', 'r') as f:
    codebook = json.load(f)
    
codebook_str = json.dumps(codebook, indent=2, ensure_ascii=False)


In [ ]:
#helper functions

##parsing responses from model
def parse_json_with_fallback(content_str):
    # Strip whitespace
    content_str = content_str.strip()
    # If the string is supposed to end with '}', but doesn't, add it.
    if not content_str.endswith('}'):
        content_str += '}'
    # Now try parsing
    try:
        dict_val = json.loads(content_str)
        str_val = dict_val.get('Relevance')
        binary_val = 1 if str_val.strip().lower() == "yes" else 0

        return binary_val
    except json.JSONDecodeError:
        return pd.NA



##compute kappa and accuracy
def compute_scores(df_pred, df_gold, column_pred, column_gold, column_match):
    df_pred_reduced = df_pred[[column_match, column_pred]]
    df_gold_reduced = df_gold[[column_match, column_gold]]

    #join these two dataframes
    df_temp = (
        df_pred_reduced
            .merge(df_gold_reduced, on=column_match, how='inner')   # keep only matching IDs
            .dropna(subset=[column_pred, column_gold])          # drop rows where values are NaN
            .reset_index(drop=True)                                 # tidy up the index
    )
    
    #ensure that the columns have data in the same type 
    df_temp[column_pred] = df_temp[column_pred].astype(int)
    df_temp[column_gold] = df_temp[column_gold].astype(int)

    #compute scores
    acc = round(100*sk.metrics.accuracy_score(df_temp[column_gold], df_temp[column_pred]), 3)
    k = round(sk.metrics.cohen_kappa_score(df_temp[column_gold], df_temp[column_pred]), 3)

    print("Accuracy is " + str(acc) + "%, and Cohen's kappa is " + str(k))
    

In [ ]:
#Zero Shot relevance binary classification - with codebook

#Prompts
SYSTEM_PROMPT = """You're a communication researcher who is studying the news reporting of Mpox. You’ll perform relevance coding on news articles, by classifying articles as relevant, or not relevant. 
Remember to prioritize accuracy and clarity in your analysis, using the provided context and your expertise to guide your evaluation
Here's the codebook for you to follow for class definitions: """ + codebook_str


USER_PROMPT =  """Is the following article relevant?  Provide your response in a JSON array format, as follows, and include nothing else in the response : { "Relevance": "yes/no" }. 
If you are uncertain about the classification, force a decision to choose "yes" or "no". """

#sample for testing things first
#df_test = df.sample(n= 5, random_state= 42).reset_index()

df_test = df

df1 = df_test

predictions = list()

for i in range(len(df1)):
    try:
        messages = [
        {
         "role": "user", 
         "content": USER_PROMPT + df1["text"][i] 
        }

            ]
    
        outputs = client.messages.create(
                model=model_name,
                max_tokens=256,
                temperature=0,  #Set to 0 for more deterministic outputs
                system=SYSTEM_PROMPT,
                messages=messages)

        predictions.append(outputs.content[0].text)
    
    #if there is an error
    except Exception as e:
            predictions.append(None)

    
    if(i%10 == 0): print(str(i) + " iterations finished")

#save the responses
RelevanceList = []

for output in predictions:
    content_str = output
    
    annotation = parse_json_with_fallback(content_str)

    RelevanceList.append(annotation)

df1["relevance"] = RelevanceList

#compute metrics
compute_scores(df_gold= df_gold, df_pred = df1, column_match= 'stories_id', column_gold= 'gold', column_pred= 'relevance')

df1.to_csv("Predicted/" + model_name + "_zshot_withCB.csv")